# Colab bootstrap

See `docs/colab.md` for the full workflow, checkpoint locations, and how to resume after a disconnect.

**Cell 6 ("Run the experiment") is the only thing you edit between experiments** -- everything else stays fixed.

A version mismatch against `requirements-colab.txt`'s pins invalidates any direct comparison to the epoch-39 baseline (oil IoU 0.1057 / 0.1074 re-verified) -- see that file's header comment.

## 0. Options

Edit `FORCE_FRESH_DOWNLOAD` here if needed -- everything else in this section is fixed.

In [ ]:
# True: ignore any cached dataset archive on Drive and re-download from Zenodo
# from scratch (also re-writes the Drive cache afterward). Use this if the
# download/extract scripts change, or if you suspect the cached archive is
# stale/corrupt -- there is no automatic staleness check, this flag is it.
FORCE_FRESH_DOWNLOAD = False

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Mount Drive

Used for the protected baseline checkpoint, the dataset cache (step 5), and persisting `output_dir` (checkpoints/metrics) across disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/sih26143-oil-spill-attribution'
!mkdir -p "$DRIVE_PROJECT_DIR/checkpoints" "$DRIVE_PROJECT_DIR/experiments" "$DRIVE_PROJECT_DIR/dataset_cache"

## 3. Clone the repo

In [ ]:
REPO_URL = 'https://github.com/Rahil-Mokashi/sih-initial.git'
BRANCH = 'main'

!git clone -b $BRANCH $REPO_URL /content/repo
%cd /content/repo

## 4. Install pinned dependencies

In [ ]:
!pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q -r requirements-colab.txt

## 5. Data + protected checkpoint

**Cache-first**: checks Drive for a previously-built dataset archive (`dataset_cache/zenodo_extracted.tar`) and extracts from there if present and `FORCE_FRESH_DOWNLOAD` is `False`; only downloads from Zenodo if the cache is missing or the flag forces it. After a fresh download, the extracted result is tarred and written back to Drive so the NEXT session (e.g. after a disconnect mid-run) doesn't pay the download+extract cost again.

What's actually cached: the extracted per-image GeoTIFFs (`data/raw/*/images_extracted`, `masks_extracted`) and the `data/processed/{train,val}_manifest.csv` manifests `build_training_pool.py` produces from them -- NOT a separate "tiles" artifact, since none is ever materialized to disk (`ZenodoTileDataset` reads 512x512 windows on the fly at train time; caching the source images is the equivalent reusable unit).

**Read `docs/colab.md`'s "Dataset caching" section before relying on this** -- the cache archive is large (comparable to the ~94GB combined Part I/II/III extracted size) and free Google Drive accounts only get 15GB, so this may not fit without paid Drive storage. Cell output below reports the real elapsed time and data volume either way, precisely so you can decide whether caching is worth it for your account instead of guessing.

**Checkpoint**: the protected `checkpoints/baseline_epoch39/` copy is `checkpoints/**/*.pt`-gitignored (172MB, too large for a normal git push) -- upload it to Drive once yourself (matching `checkpoints/baseline_epoch39/MANIFEST.json`'s sha256 so you know it's the right file); this cell copies it into the clone if present. Harmless to leave in place if your experiment's config sets `resume_from: null`.

In [ ]:
import os, shutil, subprocess, tarfile, time
from pathlib import Path

REPO_DIR = Path('/content/repo')
CACHE_ARCHIVE_DRIVE = Path(DRIVE_PROJECT_DIR) / 'dataset_cache' / 'zenodo_extracted.tar'
CACHE_ARCHIVE_LOCAL = Path('/content/zenodo_extracted.tar')

# What this cache actually covers -- see the markdown cell above.
CACHED_PATHS = [
    REPO_DIR / 'data' / 'raw' / 'zenodo_sar_oil_spill' / 'images_extracted',
    REPO_DIR / 'data' / 'raw' / 'zenodo_sar_oil_spill' / 'masks_extracted',
    REPO_DIR / 'data' / 'raw' / 'zenodo_sar_oil_spill_part2' / 'images_extracted',
    REPO_DIR / 'data' / 'raw' / 'zenodo_sar_oil_spill_part2' / 'masks_extracted',
    REPO_DIR / 'data' / 'raw' / 'zenodo_sar_oil_spill_part3' / 'images_extracted',
    REPO_DIR / 'data' / 'raw' / 'zenodo_sar_oil_spill_part3' / 'masks_extracted',
    REPO_DIR / 'data' / 'processed' / 'train_manifest.csv',
    REPO_DIR / 'data' / 'processed' / 'val_manifest.csv',
]


def dir_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    if path.is_file():
        return path.stat().st_size
    return sum(f.stat().st_size for f in path.rglob('*') if f.is_file())


def human(n_bytes: float) -> str:
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n_bytes < 1024:
            return f'{n_bytes:.1f}{unit}'
        n_bytes /= 1024
    return f'{n_bytes:.1f}PB'


t_start = time.time()

if not FORCE_FRESH_DOWNLOAD and CACHE_ARCHIVE_DRIVE.exists():
    cache_size = CACHE_ARCHIVE_DRIVE.stat().st_size
    print(f'found Drive cache: {CACHE_ARCHIVE_DRIVE} ({human(cache_size)}) -- extracting instead of downloading')

    t_copy = time.time()
    shutil.copy(CACHE_ARCHIVE_DRIVE, CACHE_ARCHIVE_LOCAL)
    print(f'  copied from Drive: {time.time() - t_copy:.1f}s')

    t_extract = time.time()
    with tarfile.open(CACHE_ARCHIVE_LOCAL) as tar:
        tar.extractall('/')
    print(f'  extracted: {time.time() - t_extract:.1f}s')

    data_volume = cache_size
    source = 'Drive cache'
else:
    print('FORCE_FRESH_DOWNLOAD=True -- skipping cache check' if FORCE_FRESH_DOWNLOAD
          else f'no Drive cache at {CACHE_ARCHIVE_DRIVE} -- downloading from Zenodo')

    raw_dir = REPO_DIR / 'data' / 'raw'
    size_before = dir_size_bytes(raw_dir)

    for script in [
        'scripts/download_zenodo_sample.py',
        'scripts/download_zenodo_part2_parallel.py',
        'scripts/download_zenodo_part2_part3.py',
        'scripts/extract_zenodo_part2.py',
        'scripts/extract_zenodo_part3.py',
        'scripts/build_training_pool.py',
    ]:
        t_step = time.time()
        subprocess.run(['python', script], cwd=REPO_DIR, check=True)
        print(f'  {script}: {time.time() - t_step:.1f}s')

    data_volume = dir_size_bytes(raw_dir) - size_before
    source = 'Zenodo (fresh download)'

    print(f'writing Drive cache for next time: {CACHE_ARCHIVE_DRIVE}')
    t_tar = time.time()
    CACHE_ARCHIVE_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(CACHE_ARCHIVE_LOCAL, 'w') as tar:  # no compression -- SAR float32 GeoTIFFs don't shrink, gzip would just cost CPU time
        for p in CACHED_PATHS:
            if p.exists():
                tar.add(p, arcname=str(p.relative_to('/')))
    print(f'  tar created: {time.time() - t_tar:.1f}s ({human(CACHE_ARCHIVE_LOCAL.stat().st_size)})')

    t_upload = time.time()
    CACHE_ARCHIVE_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(CACHE_ARCHIVE_LOCAL, CACHE_ARCHIVE_DRIVE)
    print(f'  uploaded to Drive: {time.time() - t_upload:.1f}s')

elapsed = time.time() - t_start
print(f'\n=== cell 5 summary: source={source}, elapsed={elapsed:.1f}s ({elapsed/60:.1f} min), data volume={human(data_volume)} ===')
print('Record this in docs/colab.md / LOG.md the first time you run each path (cache-hit vs. fresh-download) so future sessions know what to expect.')

In [ ]:
drive_ckpt = f'{DRIVE_PROJECT_DIR}/checkpoints/baseline_epoch39/unet_resnet18_epoch39.pt'
local_ckpt_dir = '/content/repo/checkpoints/baseline_epoch39'
os.makedirs(local_ckpt_dir, exist_ok=True)
if os.path.exists(drive_ckpt):
    shutil.copy(drive_ckpt, f'{local_ckpt_dir}/unet_resnet18_epoch39.pt')
    print('copied protected baseline checkpoint from Drive')
else:
    print(f'WARNING: {drive_ckpt} not found on Drive -- upload it there first if this experiment needs resume_from')

## 6. Run the experiment

**This is the only cell you edit between experiments.** `output_dir` in the config should point under `/content/drive/...` (or be copied there after each epoch) so checkpoints/metrics survive a disconnect -- see docs/colab.md.

In [ ]:
EXPERIMENT_NAME = "baseline"
!python train.py --config configs/{EXPERIMENT_NAME}.yaml